---

# PIVOT TABLES

---

The `.pivot_table()` Method let's you create Excel-style **pivot tables**

- `index:` returns a row index with distinct values from the specified column
- `columns:` returns a column index with distinct values from the specified column
- `values:` the column, or columns to perform the **aggregations on**.
- `aggfun:` defines the aggregatrion function, or functions, to perform on the "***values***"
- `margins:` returns row and column totals when TRUE (***False by default***)

<br>

In [1]:
import numpy as np
import pandas as pd

In [2]:
retail = pd.read_csv('Data/retail_2016_2017.csv')
retail.head()

,id,date,store_nbr,family,sales,onpromotion
0,1945944,2016-01-01,1,AUTOMOTIVE,0.0,0
1,1945945,2016-01-01,1,BABY CARE,0.0,0
2,1945946,2016-01-01,1,BEAUTY,0.0,0
3,1945947,2016-01-01,1,BEVERAGES,0.0,0
4,1945948,2016-01-01,1,BOOKS,0.0,0


In [3]:
reduced_retail = (retail[(retail.store_nbr <= 6) & 
    (retail.family.isin(['AUTOMOTIVE', 
                         'BOOKS', 
                         'HARDWARE', 
                         'MAGAZINES']))])
reduced_retail

,id,date,store_nbr,family,sales,onpromotion
0,1945944,2016-01-01,1,AUTOMOTIVE,0.0,0
4,1945948,2016-01-01,1,BOOKS,0.0,0
14,1945958,2016-01-01,1,HARDWARE,0.0,0
23,1945967,2016-01-01,1,MAGAZINES,0.0,0
363,1946307,2016-01-01,2,AUTOMOTIVE,0.0,0
...,...,...,...,...,...,...
1054637,3000581,2017-08-15,5,MAGAZINES,15.0,0
1054812,3000756,2017-08-15,6,AUTOMOTIVE,7.0,0
1054816,3000760,2017-08-15,6,BOOKS,0.0,0
1054826,3000770,2017-08-15,6,HARDWARE,1.0,0


---

## 1. Basic Pivot Tables

In [4]:
(reduced_retail
    .pivot_table(index='family', columns='store_nbr', values='sales', 
                 aggfunc='sum', margins=True, margins_name='TOTAL')
    .round())

store_nbr,1,2,3,4,5,6,TOTAL
family,,,,,,,
AUTOMOTIVE,2524.0,3918.0,6790.0,2565.0,3667.0,3442.0,22906.0
BOOKS,211.0,239.0,540.0,266.0,230.0,76.0,1562.0
HARDWARE,1056.0,904.0,2034.0,923.0,619.0,674.0,6210.0
MAGAZINES,4095.0,5126.0,15477.0,4572.0,4774.0,4736.0,38780.0
TOTAL,7886.0,10187.0,24841.0,8326.0,9290.0,8928.0,69458.0


---

## LABORATORY

In [5]:
league = pd.read_excel('Data/premier_league_games_full.xlsx')
league.head(3)

,id,league_name,season,HomeTeam,AwayTeam,HomeGoals,AwayGoals
0,1729,England Premier League,2008/2009,Manchester United,Newcastle United,1,1
1,1730,England Premier League,2008/2009,Arsenal,West Bromwich Albion,1,0
2,1731,England Premier League,2008/2009,Sunderland,Liverpool,0,1


In [6]:
league.query(
    "HomeTeam in ['Manchester City', 'Manchester United', 'Chelsea', 'Arsenal', 'Liverpool', 'Everton']"
            ).pivot_table(index='HomeTeam', 
                   columns='season', 
                   values='HomeGoals', 
                   aggfunc='sum',
                   fill_value=0,
                   margins=True, 
                   margins_name='TOTAL').round(2)

season,2008/2009,2009/2010,2010/2011,2011/2012,2012/2013,2013/2014,2014/2015,2015/2016,TOTAL
HomeTeam,,,,,,,,,
Arsenal,31,48,33,39,47,36,41,31,306
Chelsea,33,68,39,41,41,43,36,32,333
Everton,31,35,31,28,33,38,27,35,258
Liverpool,41,43,37,24,33,53,30,33,294
Manchester City,40,41,34,55,41,63,44,47,365
Manchester United,43,52,49,52,45,29,41,27,338
TOTAL,219,287,223,239,240,262,219,205,1894


---

## 2. Multiple Aggregation Pivot Tables

In [7]:
(reduced_retail.pivot_table(index='family', 
                           columns='store_nbr', 
                           values='sales',
                           aggfunc=('min', 'max'),
                           ))

max                                min                         
store_nbr      1     2     3     4     5     6    1    2    3    4    5    6
family                                                                      
AUTOMOTIVE  19.0  23.0  32.0  19.0  18.0  24.0  0.0  0.0  0.0  0.0  0.0  0.0
BOOKS        8.0   9.0  11.0   9.0   6.0   6.0  0.0  0.0  0.0  0.0  0.0  0.0
HARDWARE    13.0   9.0  15.0  10.0   6.0   8.0  0.0  0.0  0.0  0.0  0.0  0.0
MAGAZINES   26.0  32.0  79.0  32.0  27.0  28.0  0.0  0.0  0.0  0.0  0.0  0.0

In [13]:
# EXAMPLE NO.2
(reduced_retail.pivot_table(index='family', columns='store_nbr',
                           aggfunc=({'sales':['sum', 'mean'],
                                     'onpromotion':['max']}))
)

onpromotion                    sales                       \
                   max                     mean                        
store_nbr            1  2  3  4  5  6         1         2          3   
family                                                                 
AUTOMOTIVE           1  1  1  1  2  1  4.263514  6.618243  11.469595   
BOOKS                0  0  0  0  0  0  0.356419  0.403716   0.912162   
HARDWARE             0  0  1  0  1  0  1.783784  1.527027   3.435811   
MAGAZINES            0  0  0  0  1  0  6.917230  8.658784  26.143581   

                                                                           \
                                             sum                            
store_nbr          4         5         6       1       2        3       4   
family                                                                      
AUTOMOTIVE  4.332770  6.194257  5.814189  2524.0  3918.0   6790.0  2565.0   
BOOKS       0.449324  0.388514  0.128378   211.0   239.0    540.0   266.0   
HARDWARE    1.559122  1.045608  1.138514  1056.0   904.0   2034.0   923.0   
MAGAZINES   7.722973  8.064189  8.000000  4095.0  5126.0  15477.0  4572.0   

                            
                            
store_nbr        5       6  
family                      
AUTOMOTIVE  3667.0  3442.0  
BOOKS        230.0    76.0  
HARDWARE     619.0   674.0  
MAGAZINES   4774.0  4736.0

---

## 3. Pivot Table VS. GroupBy (*When to Use each*)

If the column argument isn't specified in a `pivot table`, it will return a table that's identical to one grouped by the index columns

---

- `PRO TIP`: Use `.groupby()` method if you don't need columns in the pivot, as you can use named aggregations to flatten the column index

In [20]:
(reduced_retail
    .query("store_nbr in [1, 2, 3]")
    .pivot_table(index=['family', 'store_nbr'], 
                 aggfunc=({'sales':['sum', 'mean'],
                           'onpromotion':['sum', 'mean']}))
)

onpromotion          sales         
                            mean sum       mean      sum
family     store_nbr                                    
AUTOMOTIVE 1            0.023649  14   4.263514   2524.0
           2            0.020270  12   6.618243   3918.0
           3            0.020270  12  11.469595   6790.0
BOOKS      1            0.000000   0   0.356419    211.0
           2            0.000000   0   0.403716    239.0
           3            0.000000   0   0.912162    540.0
HARDWARE   1            0.000000   0   1.783784   1056.0
           2            0.000000   0   1.527027    904.0
           3            0.001689   1   3.435811   2034.0
MAGAZINES  1            0.000000   0   6.917230   4095.0
           2            0.000000   0   8.658784   5126.0
           3            0.000000   0  26.143581  15477.0

In [37]:
(reduced_retail
    .query("store_nbr in [1, 2, 3]")
    .groupby(['family', 'store_nbr'])
    .agg({'sales':['sum', 'mean'], 'onpromotion':['sum', 'mean']}))

sales            onpromotion          
                          sum       mean         sum      mean
family     store_nbr                                          
AUTOMOTIVE 1           2524.0   4.263514          14  0.023649
           2           3918.0   6.618243          12  0.020270
           3           6790.0  11.469595          12  0.020270
BOOKS      1            211.0   0.356419           0  0.000000
           2            239.0   0.403716           0  0.000000
           3            540.0   0.912162           0  0.000000
HARDWARE   1           1056.0   1.783784           0  0.000000
           2            904.0   1.527027           0  0.000000
           3           2034.0   3.435811           1  0.001689
MAGAZINES  1           4095.0   6.917230           0  0.000000
           2           5126.0   8.658784           0  0.000000
           3          15477.0  26.143581           0  0.000000